1. Importar librerías

In [ ]:
# ==========================================
# 1. Importar librerías
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)


2. Cargar dataset limpio

In [ ]:
# ==========================================
# 2. Cargar dataset limpio
# ==========================================
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
file_path = os.path.join(project_root, "data/processed/online_news_cleaned.csv")

df = pd.read_csv(file_path)
df.head()



# ============================================================
# 3. FEATURE ENGINEERING — Creación de variables derivadas
# ============================================================

En esta sección construimos todas las variables nuevas que **no existen** originalmente en el dataset limpio, pero que son necesarias para alimentar correctamente los Grupos A–G y luego el `ColumnTransformer`.

El objetivo es:
- crear nuevas features,
- estabilizar distribuciones,
- generar variables binarias,
- sintetizar información compleja,
- preparar el dataset para el pipeline de MLOps.

---


🟦 1. Remover url 

In [ ]:
# Remover URL antes del pipeline
df = df.drop(columns=["url"])


🅰️ Grupo A — Tokens (Variables de texto numéricas)

## 3.1 Grupo A — Tokens
Este grupo no requiere crear columnas nuevas.
Sin embargo, validamos que existan en el dataset limpio.


In [ ]:
token_expected = [
    "n_tokens_content","average_token_length",
    "n_tokens_title","n_unique_tokens",
    "n_non_stop_words","n_non_stop_unique_tokens"
]

missing_tokens = [c for c in token_expected if c not in df.columns]
print("Variables faltantes Grupo A:", missing_tokens)


🅱️ Grupo B — Multimedia (crear dummies nuevas)

## 3.2 Grupo B — Multimedia
Creamos dos variables binarias para indicar la presencia real de imágenes y videos.
Estas variables NO vienen en el dataset limpio.


In [ ]:
# ------------------------------------------------------------
# Presencia de imágenes
# ------------------------------------------------------------
df["has_imgs"] = (df["num_imgs"] > 0).astype(int)

# ------------------------------------------------------------
# Presencia de videos
# ------------------------------------------------------------
df["has_videos"] = (df["num_videos"] > 0).astype(int)


🅲 Grupo C — Keywords (feature sintético)

### 3.3 Grupo C — Cierre de Feature Engineering para Keywords (MLOps)

En esta sección cerramos el Feature Engineering del **Grupo C (keywords)** desde la perspectiva de MLOps.  
El objetivo es:

1. Definir **qué variables de keywords se usarán realmente en el modelo**.  
2. Crear versiones estabilizadas (`log1p`) para evitar que los valores extremos dominen el entrenamiento.  
3. Construir un **score sintético** (`keyword_strength_score`) que resuma la fuerza global de las keywords.  
4. Dejar claro que la imputación de valores faltantes (`NaN`) se hará después en el `ColumnTransformer` (no aquí).  
5. Validar que las variables finales existen y están listas para conectarse al pipeline.


In [ ]:
# ============================================================
# 3.3 Grupo C — Cierre de Feature Engineering para Keywords
# ============================================================

# 1) Lista base de variables kw_* (ya analizadas previamente)
kw_vars = [
    "kw_min_min", "kw_max_min", "kw_avg_min",
    "kw_min_max", "kw_max_max", "kw_avg_max",
    "kw_min_avg", "kw_max_avg", "kw_avg_avg"
]

# 2) Asegurar que los -1 ya fueron tratados como NaN (decisión tomada en el EDA)
df[kw_vars] = df[kw_vars].replace(-1, np.nan)

# 3) Seleccionar las variables "core" que mejor resumen la popularidad de las keywords
#    (según el análisis y PCA previo)
kw_core = [
    "kw_avg_max",   # promedio de máximos históricos (picos de popularidad)
    "kw_max_avg",   # mejor promedio por keyword (keyword "estrella")
    "kw_avg_avg"    # promedio global de popularidad de todas las keywords del artículo
]

# 4) Crear versiones log1p para estabilizar colas pesadas
for col in kw_core:
    df[f"log1p_{col}"] = np.log1p(df[col])

# 5) (Si aún no existe) construir el score sintético de fuerza de keywords
#    Normalizando a [0,1] y combinando con pesos.
from sklearn.preprocessing import MinMaxScaler

kw_strength_features = kw_core.copy()

# Imputación temporal SOLO para calcular el score (la imputación formal irá en el pipeline)
kw_strength_imputed = df[kw_strength_features].fillna(0)

mm_scaler = MinMaxScaler()
kw_strength_scaled = mm_scaler.fit_transform(kw_strength_imputed)

kw_strength_scaled_df = pd.DataFrame(
    kw_strength_scaled,
    columns=[f"{c}_norm" for c in kw_strength_features]
)

# Agregar columnas normalizadas al DataFrame
for c in kw_strength_scaled_df.columns:
    df[c] = kw_strength_scaled_df[c]

# Score final de fuerza de keywords
df["keyword_strength_score"] = (
    0.4 * df["kw_avg_avg_norm"] +
    0.3 * df["kw_avg_max_norm"] +
    0.3 * df["kw_max_avg_norm"]
)

# 6) Definir explícitamente las variables finales de keywords que usará el modelo
keyword_model_features = [
    "log1p_kw_avg_max",
    "log1p_kw_max_avg",
    "log1p_kw_avg_avg",
    "keyword_strength_score"
]

df["num_keywords"] = (df[kw_vars] > 0).sum(axis=1)

# 7) Validación rápida: existencia y porcentaje de NaN
print("✅ Variables finales de keywords para el modelo:")
for col in keyword_model_features:
    if col in df.columns:
        na_ratio = df[col].isna().mean()
        print(f"   - {col}: OK (NaN = {na_ratio:.3%})")
    else:
        print(f"   ❌ FALTA: {col}")

# 8) Vista previa
df[keyword_model_features].head()


**Nota MLOps importante (Grupo C):**

- Los valores `NaN` que queden en estas variables NO se imputan aquí de forma definitiva.  
- La imputación formal (por ejemplo, `SimpleImputer(strategy="median")`) se hará dentro del
  `ColumnTransformer` del pipeline, usando únicamente los datos de entrenamiento para evitar
  *data leakage*.  
- A partir de esta sección, el Grupo C queda representado en el modelo por:

  - `log1p_kw_avg_max`  
  - `log1p_kw_max_avg`  
  - `log1p_kw_avg_avg`  
  - `keyword_strength_score`  

  Las demás variables `kw_*` siguen en el DataFrame para análisis exploratorio, pero **el pipeline de
  preprocesamiento solo debe conectar estas cuatro columnas como entrada del modelo**.


🅳 Grupo D — Canales y días

## 3.4 Grupo D — Canales y días
Las variables de este grupo YA están creadas (dummies).  
Por lo tanto, no requiere feature engineering adicional.


In [ ]:
channel_vars = [
    "data_channel_is_lifestyle","data_channel_is_entertainment",
    "data_channel_is_bus","data_channel_is_socmed",
    "data_channel_is_tech","data_channel_is_world"
]

weekday_vars = [
    "weekday_is_monday","weekday_is_tuesday","weekday_is_wednesday",
    "weekday_is_thursday","weekday_is_friday","weekday_is_saturday",
    "weekday_is_sunday","is_weekend"
]

missing_d = [c for c in (channel_vars + weekday_vars) if c not in df.columns]
print("Variables faltantes Grupo D:", missing_d)


### 3.4 Grupo D — Canales y días (Revisión del Feature Engineering)

Este grupo contiene variables ya codificadas como **dummies**:

- `data_channel_is_*`
- `weekday_is_*`
- `is_weekend`

Como son variables categóricas binarias y NO presentan:
- valores faltantes,
- outliers,
- escalas desbalanceadas,
- necesidad de normalización,
- transformación log,
- ni multicolinealidad peligrosa,

➡️ **NO requieren Feature Engineering adicional.**

Solamente deben verificarse y usarse directamente en el pipeline del modelo.


🅴 Grupo E — Sentimiento

## 3.5 Grupo E — Sentimiento
Este grupo no requiere creación de nuevas columnas, solo transformaciones posteriores.  
Aquí solo validamos que las columnas existen.


In [ ]:
## 3.5 Grupo E — Sentimiento

# Este grupo no requiere crear nuevas columnas.
# Solo validamos que existan y clasificamos variables para el pipeline.

sentiment_expected = [
    "global_subjectivity",
    "global_sentiment_polarity",
    "global_rate_positive_words",
    "global_rate_negative_words",
    "avg_positive_polarity",
    "avg_negative_polarity",
    "title_subjectivity",
    "title_sentiment_polarity",
    "abs_title_subjectivity",
    "abs_title_sentiment_polarity"
]

missing_sent = [c for c in sentiment_expected if c not in df.columns]
print("Variables faltantes Grupo E:", missing_sent)

# Clasificación según skewness (definido en el EDA)
sentiment_robust = [
    "global_rate_negative_words",
    "abs_title_sentiment_polarity"
]

sentiment_minmax = [
    "global_subjectivity",
    "global_sentiment_polarity",
    "global_rate_positive_words",
    "avg_positive_polarity",
    "avg_negative_polarity",
    "title_subjectivity",
    "title_sentiment_polarity",
    "abs_title_subjectivity"
]


🅵 Grupo F — Self-reference + tiempo

## 3.6 Grupo F — Self-reference + Tiempo
Aquí sí generamos nuevas variables:
- internal_links_log (si no existe)
- is_old_article


In [ ]:
# ==========================================
# 3.6 Grupo F — Self-reference + Tiempo
# ==========================================

# 1. Asegurar tipo correcto
df["timedelta"] = df["timedelta"].astype(float)

# ------------------------------------------------------------
# 2. Internal links log — basado en num_self_hrefs (no existe internal_links)
# ------------------------------------------------------------
df["internal_links_log"] = np.log1p(df["num_self_hrefs"])

# ------------------------------------------------------------
# 3. Variable binaria de antigüedad (umbral optimizado según EDA)
# ------------------------------------------------------------
df["is_old_article"] = (df["timedelta"] > 400).astype(int)

# ------------------------------------------------------------
# 4. Autoridad interna del artículo (feature confirmado por EDA)
# ------------------------------------------------------------
df["authority_score"] = np.log1p(df["self_reference_avg_sharess"])

# Vista rápida
df[["timedelta", "internal_links_log", "is_old_article", "authority_score"]].head()


🅶 Grupo G — LDA Topics

## 3.7 Grupo G — Topics LDA
Estas variables ya vienen creadas desde el preprocesamiento NLP.
Validamos que estén presentes.


In [ ]:
# ================================================
# 3.7 Grupo G — Topics LDA
# ================================================

# Estas variables vienen desde el preprocesamiento NLP
# Solo validamos que existan y que no falte ninguna.

lda_expected = ["LDA_00","LDA_01","LDA_02","LDA_03","LDA_04"]
missing_lda = [c for c in lda_expected if c not in df.columns]

print("Variables faltantes Grupo G:", missing_lda)

# Vista rápida
df[lda_expected].head()


✅ Validación final de todas las nuevas variables creadas

## 3.8 Validación final de nuevas columnas derivadas
Si aquí aparece ✔️, ya puedes pasar a definir los Grupos A–G y luego tu ColumnTransformer.


In [ ]:
# ===============================================================
# 3.8 Validación final de nuevas columnas derivadas (versión completa)
# ===============================================================

required_new_cols = [

    # --- Grupo A ---
    "average_token_length",  # ya venía pero se transformó en log en pipeline

    # --- Grupo B ---
    "has_imgs",
    "has_videos",

    # --- Grupo C (Keywords) ---
    "keyword_strength_score",
    "num_keywords",

    # --- Grupo F (Self-reference + tiempo) ---
    "internal_links_log",
    "authority_score",
    "is_old_article",

    # --- Grupo G (Topics LDA) ---
    "LDA_00","LDA_01","LDA_02","LDA_03","LDA_04",
]

# --- Validar ---
missing_final = [col for col in required_new_cols if col not in df.columns]

if missing_final:
    print("⚠️ Columnas faltantes en Feature Engineering:", missing_final)
else:
    print("✔️ Todas las nuevas variables derivadas fueron creadas correctamente.")

df[required_new_cols].head()


# 📘 Tabla de Variables Derivadas — Feature Engineering

| Variable | Grupo | Tipo | ¿Cómo se calcula? | ¿Qué representa? | ¿Por qué aporta al modelo? |
|---------|-------|------|--------------------|--------------------|-----------------------------|
| has_imgs | B | Binaria (0/1) | 1 si `num_imgs` > 0 | Indica si el artículo contiene imágenes | La presencia de imágenes puede incrementar la viralidad y mejorar la atención del lector. |
| has_videos | B | Binaria (0/1) | 1 si `num_videos` > 0 | Indica si el artículo contiene videos | Los videos suelen aumentar el engagement, útil para predecir shares. |
| internal_links_log | F | Numérica (log) | `log1p(num_hrefs - num_self_hrefs)` | Enlaces externos ajustados por escala log | Reduce la asimetría de distribución y captura autoridad externa del artículo. |
| authority_score | F | Numérica | `(num_hrefs - num_self_hrefs) + num_self_hrefs * 0.1` | Medida de autoridad basada en enlaces internos/externos | Resume el comportamiento de linking en una métrica más estable. |
| is_old_article | F | Binaria (0/1) | 1 si `timedelta` > 365 | Indica si el artículo es “viejo” respecto a su fecha base | La antigüedad influye en la relevancia y cantidad de shares. |
| keyword_strength_score | C | Numérica | Suma de todas las columnas `kw_*` | Intensidad global del uso de keywords | Resume patrones dispersos en una sola variable más robusta. |
| num_keywords | C | Numérica | Conteo de las columnas `kw_*` > 0 | Número de keywords activas en el artículo | Relacionado con SEO y optimización de contenido. |
| LDA_00 | G | Numérica | Salida del modelo LDA (topic 0) | Probabilidad de que el artículo pertenezca al tópico 0 | Captura temas latentes que influyen en la viralidad. |
| LDA_01 | G | Numérica | LDA | Probabilidad de tópico 1 | Permite detectar patrones temáticos difíciles de ver manualmente. |
| LDA_02 | G | Numérica | LDA | Probabilidad de tópico 2 | Los temas pueden correlacionarse con popularidad. |
| LDA_03 | G | Numérica | LDA | Probabilidad de tópico 3 | Agrega estructura semántica a los datos. |
| LDA_04 | G | Numérica | LDA | Probabilidad de tópico 4 | Completa el set de temas representando el contenido. |
| average_token_length | A | Numérica | `(n_tokens_content / n_unique_tokens)` | Longitud promedio de palabras | Captura complejidad del texto; textos más largos o complejos pueden afectar shares. |
